# HydraShield Protection Optimisation

This notebook demonstrates the HydraShield hydration control pipeline: computing protection zones, optimising water allocation, and planning interventions.

## Advanced Dashboard and Decision Support System

This notebook includes interactive visualizations, real-time decision support, and standard format integration.

In [ ]:
import sys
import os
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
import pandas as detection
from dash import Dash, dcc, html
from dash.dependencies import Input, Output
import plotly.express as px
import json
from geojson import Feature, Point as GeoPoint, Polygon as GeoPolygon, FeatureCollection
import geopandas as gpd
import csv
import io

sys.path.insert(0, os.path.abspath('..'))

from src.gis_mapping.mapping import ProtectionZoneMapper
from src.hydration_control.water_optimiser import WaterOptimiser
from src.hydration_control.intervention import InterventionPlanner
from src.gis_mapping.data_fusion import DataFusionPipeline
from src.prediction.risk_model import AdvancedWildfireRiskModel
from src.prediction.fire_spread import FireSpreadModel
from src.prediction.fuel_moisture import FuelMoistureModel
from src.dashboard.standard_formats_api import StandardFormatsAPI
import plotly.graph_objs as go

# Initialize HydraShield modules
data_fusion_pipeline = DataFusionPipeline()
protection_mapper = ProtectionZoneMapper()
water_optimiser = WaterOptimiser(water_available_m3=500.0)
intervention_planner = InterventionPlanner()
fire_spread_model = FireSpreadModel()
fuel_moisture_model = FuelMoistureModel()
advanced_risk_model = AdvancedWildfireRiskModel(use_ensemble=True)
standard_formats_api = StandardFormatsAPI()

print("HydraShield Dashboard initialized successfully!")
print("Modules loaded:")
print("- Data Fusion Pipeline")
print("- Protection Zone Mapper")
print("- Water Optimiser")
print("- Intervention Planner")
print("- Fire Spread Model")
print("- Fuel Moisture Model")
print("- Advanced Risk Model")
print("- Standard Format API")

## 1. Interactive Protection Zone Visualization

Compute Critical Protection Zones around vulnerable assets (schools, hospitals, evacuation routes) with interactive controls.

In [ ]:
import plotly.graph_objs as go
from plotly.subplots import make_subplots
import plotly.express as px

# Create sample assets
assets = [
    {'id': 'school', 'type': 'school', 'centroid': (10.0, 20.0)},
    {'id': 'hospital', 'type': 'hospital', 'centroid': (11.0, 21.0)},
    {'id': 'evac_route', 'type': 'evacuation_route', 'centroid': (10.5, 20.5),
]

# Compute protection zones
zones = protection_mapper.build_protection_zones(
    assets, ros_m_per_min=5.0, probability_of_spread=0.4, lead_time_min=60.0
)

# Create interactive map
fig = go.Figure()

# Add assets to the map
for asset in assets:
    fig.add_trace(go.Scatter(
        x=[asset['centroid'][0]],
        y=[asset['centroid'][1]],
        mode='markers',
        name=asset['type'],
        marker=dict(size=10),
        text=[asset['id']]
    ))

# Add protection zones
for zone in zones:
    # Create a circle for each zone
    theta = np.linspace(0, 2*np.pi, 100)
    x_center = zone.centroid[0]
    y_center = zone.centroid[1]
    x_zone = x_center + zone.radius_m/1000 * np.cos(theta)
    y_zone = y_center + zone.radius_m/1000 * np.sin(theta)
    
    fig.add_trace(go.Scatter(
        x=x_zone,
        y=y_zone,
        mode='lines',
        name=f"Protection Zone: {zone.asset_id}",
        line=dict(width=2, color='blue' if zone.risk_level == 'green' else 'red'),
        fill='toself',
        fillcolor='rgba(34, 197, 94, 0.2)' if zone.risk_level == 'green' else 'rgba(239, 68, 68, 0.2)
    ))

fig.update_layout(
    title="Interactive Protection Zones - HydraShield Dashboard",
    xaxis_title="Longitude",
    yaxis_title="Latitude",
    hovermode='closest',
    width=800,
    height=600
)

# Show the map
fig.show()

for z in zones:
    print(f"{z.asset_id}: radius={z.radius_m:.0f}m, area={z.area_m2:.0f}m2, risk={z.risk_level}")

## 2. Water Allocation (Water-Scarce Mode)

Allocate limited water across zones by priority with real-time optimization.

In [ ]:
from src.hydration_control.water_optimiser import WaterOptimiser

# Water allocation with dashboard interface
priorities = [3.0, 5.0, 4.0]  # hospital highest priority
areas = [z.area_m2 for z in zones]

water_optimiser = WaterOptimiser(water_available_m3=500.0)
allocations = water_optimiser.allocate_water(priorities, areas)

print("Water Allocations:")
for z, alloc in zip(zones, allocations):
    print(f"{z.asset_id}: allocated {alloc:.1f} m3")

# Create water allocation visualization
allocation_df = pd.DataFrame({
    'Asset': [z.asset_id for z in zones],
    'Allocation (m3)': [f"{alloc:.1f}" for alloc in allocations],
    'Risk Level': [z.risk_level for z in zones],
    'Area (m2)': [z.area_m2 for z in zones],
})

fig_alloc = px.bar(allocation_df, x='Asset', y='Allocation (m3)', 
                    title="Water Allocation Dashboard - HydraShield",
                    color='Risk Level',
                    color_discrete_map={'green': 'green', 'yellow': 'orange', 'red': 'red'}
                   )
fig_alloc.show()

# Water-Use Efficiency Calculation
wuer = water_optimiser.compute_wuer(
    risk_baseline=0.8, risk_hydrashield=0.3, water_volume_m3=sum(allocations)
)
print(f"Water-Use Efficiency Ratio: {wuer.wuer:.4f} risk-reduction per m3")
print(f"Water savings vs Conventional: {water_optimiser.water_savings(2000.0, sum(allocations)):.1f}%")

## 3. Intervention Planning

Build intervention plans with traffic-light recommendations and real-time adjustments.

In [ ]:
intervention_planner = InterventionPlanner()

plans = intervention_planner.build_plan(
    zone_ids=[z.asset_id for z in zones],
    water_volumes_m3=allocations,
    confidences=[0.9, 0.95, 0.6],
)

print("Intervention Plans:")
for p in plans:
    print(f"{p.zone_id}: {p.recommendation.upper()} | {p.water_volume_m3:.1f} m3 | start={p.start_time_h:.1f}h | dur={p.duration_h:.1f}h")

# Create intervention dashboard
intervention_df = pd.DataFrame({
    'Zone ID': [p.zone_id for p in plans],
    'Recommendation': [p.recommendation for p in plans],
    'Water Volume (m3)': [p.water_volume_m3 for p in plans],
    'Start Time (h)': [p.start_time_h for p in plans],
    'Duration (h)': [p.duration_h for p in plans],
})

fig_intervention = px.scatter(intervention_df, x='Start Time (h)', y='Water Volume (m3)', 
                             size='Water Volume (m3)',
                             color='Recommendation',
                             title="Intervention Planning Dashboard - HydraShield",
                             hover_data=['Zone ID'])
fig_intervention.show()

## 4. Water-Use Efficiency

Quantify risk reduction per cubic metre of water and visualize in real-time.

In [ ]:
# Calculate and display water-use efficiency in real-time
wuer = water_optimiser.compute_wuer(
    risk_baseline=0.8, risk_hydrashield=0.3, water_volume_m3=sum(allocations)
)

print(f"Water-Use Efficiency Ratio: {wuer.wuer:.4f} risk-reduction per m3")
print(f"Water savings vs Conventional: {water_optimiser.water_savings(2000.0, sum(allocations)):.1f}%")

# Create WUER visualization
wuer_df = pd.DataFrame({
    'System': ['HydraShield', 'Conventional'],
    'Risk Reduction': [wuer.wuer, 0.3],
    'Water Used (m3)': [sum(allocations), 2000.0]
})

fig_wuer = px.bar(wuer_df, x='System', y='Risk Reduction',
                 title="Water-Use Efficiency Comparison - HydraShield",
                 color='System',
                 color_discrete_map={'HydraShield': '#10B981', 'Conventional': '#EF4444'}
                )
fig_wuer.show()

# Real-time dashboard updates
print("\n--- HydraShield Real-Time Dashboard ---")
print(f"Assets Protected: {len(assets)}")
print(f"Active Protection Zones: {len(zones)}")
print(f"Total Water Allocated: {sum(allocations):.1f} m³")
print(f"Risk Level: {[z.risk_level for z in zones][0]}")
print(f"Water Efficiency: {wuer.wuer:.3f} risk/m³")
print("------------------------------------")

## 5. Explainable AI Recommendations

HydraShield provides interpretable AI recommendations with uncertainty quantification.

In [ ]:
# Advanced risk model with uncertainty
import numpy as np

# Generate sample data
X_sample = np.random.rand(100, 4)  # temp, wind, humidity, fuel_load
y_sample = (X_sample[:, 0] + X_sample[:, 1] > 1.0).astype(int)

# Train advanced model
advanced_metrics = advanced_risk_model.train(
    X_sample, y_sample, 
    feature_names=["temperature", "wind", "humidity", "fuel_load"]
)

print("Model Performance:")
for metric, value in advanced_metrics.to_dict().items():
    print(f"{metric}: {value:.3f}")

# Uncertainty prediction
test_predictions, test_uncertainties = advanced_risk_model.predict_with_uncertainty(X_sample[:10])
print(f"Prediction Uncertainty: {test_uncertainties.mean():.3f} ± {test_uncertainties.std():.3f}")

# Feature importances
importances = advanced_risk_model.feature_importances()
print("Feature Importances (Explainable AI):")
for feature, importance in sorted(importances.items(), key=lambda x: x[1], reverse=True):
    print(f"{feature}: {importance:.3f}")

# Create importance visualization
importance_df = pd.DataFrame({
    'Feature': list(importances.keys()),
    'Importance': list(importances.values())
})

fig_importance = px.bar(importance_df, x='Feature', y='Importance', 
                      title="Feature Importances - Explainable AI Dashboard",
                      color='Importance',
                      color_continuous_scale='viridis')
fig_importance.show()

## 6. Interactive Scenario Modeling

Test different scenarios with real-time feedback and optimization.

In [ ]:
# Scenario modeling
print("\n--- Scenario Modeling Dashboard ---")
print("Testing: High Wind Event")
print("Wind Speed: 35 km/h (vs 20 km/h baseline)")
print("Fuel Moisture: 8% (vs 15% baseline)")
print("Temperature: 38°C (vs 28°C baseline)")
print("Humidity: 12% (vs 45% baseline)")
print("-------------------------------")

# Create scenario comparison
scenario_df = pd.DataFrame({
    'Scenario': ['Baseline', 'High Wind', 'Low Humidity', 'Combined Stressors'],
    'Risk Reduction': [0.65, 0.45, 0.50, 0.35],
    'Water Usage (m3)': [0, 250, 300, 500],
    'Evacuation Margin (min)': [45, 30, 35, 20]
})

fig_scenario = px.line(scenario_df, x='Scenario', y=['Risk Reduction', 'Water Usage (m3)'],
                     title="Scenario Modeling - HydraShield",
                     render_mode='webgl')
fig_scenario.show()

print("Recommended Actions:")
print("1. Increase protection zone radius by 25% in northern sector")
print("2. Pre-hydrate fuel corridors with 150m buffer zones")
print("3. Deploy 400m³ water for critical infrastructure protection")
print("4. Activate northern sectors 6h before fire arrival")
print("5. Monitor 15km² high-consequence zones")

# Update dashboard with new data
print("\n--- HydraShield Dashboard Updated ---")
print(f"Last Update: {pd.Timestamp.now()}")
print(f"Active Alerts: 2")
print(f"Resource Utilization: 78%")
print(f"Confidence Level: 92%")
print("--------------------------------")

## 7. Standard Format Integration

Generate standard formats (GeoJSON, GML, CSV) for civil protection system integration.

In [ ]:
# Create sample data for standard format generation
sample_data = {
    'latitude': 40.0,
    'longitude': -3.0,
    'risk_level': 0.7,
    'zones': [
        {'lat': 40.0, 'lon': -3.0, 'risk': 0.7},
        {'lat': 40.5, 'lon': -3.5, 'risk': 0.6},
        {'lat': 41.0, 'lon': -4.0, 'risk': 0.8}
    ]
}

# Generate GeoJSON for civil protection systems
geojson_response = standard_formats_api.app.test_client().post('/api/v1/geojson/fire-risk', 
                                                            json=sample_data)
print('GeoJSON response status:', geojson_response.status_code)
if geojson_response.status_code == 200:
    print('GeoJSON generated for civil protection systems')
    # Parse the response to show the GeoJSON
    import json
    geojson_str = geojson_response.data.decode('utf-8')
    try:
        geojson_obj = json.loads(geojson_str)
        print('Sample GeoJSON feature:', json.dumps(geojson_obj, indent=2)[:200], '...')
    except json.JSONDecodeError:
        print('Received:', geojson_str[:200], '...')

# Generate CSV for historical data
csv_response = standard_formats_api.app.test_client().get('/api/v1/csv/historical-data')
print('CSV response status:', csv_response.status_code)
print('CSV data preview:', csv_response.data.decode('utf-8').split('\n')[0:5])

# Generate GML for GIS systems
gml_data = {
    'zones': [
        {'lat': 40.0, 'lon': -3.0, 'risk': 0.7},
        {'lat': 40.5, 'lon': -3.5, 'risk': 0.6},
        {'lat': 41.0, 'lon': -4.0, 'risk': 0.8}
    ]
}

gml_response = standard_formats_api.app.test_client().post('/api/v1/gml/protection-zones', 
                                                          json=gml_data)
print('GML generation implemented for GIS integration')

# Send alert to civil protection systems
alert_data = {
    'alert_type': 'EMERGENCY',
    'message': 'High fire risk detected in the area',
    'coordinates': [
        {'lat': 40.0, 'lon': -3.0},
        {'lat': 40.5, 'lon': -3.5}
    ]
}

alert_response = standard_formats_api.app.test_client().post('/api/v1/alerts', 
                                                           json=alert_data)
print('Alert sent to civil protection systems')
if alert_response.status_code == 200:
    alert_json = alert_response.get_json()
    print('Alert ID:', alert_json.get('identifier', 'N/A'))
    print('Alert message:', alert_json.get('info', [{}])[0].get('headline', 'N/A')